### Major refactoring
Since I have so many layers and parameters, I am going to need a better way to organize them. I'm also going to be adding LayerNorm so wayyyy too many parameters in general. 
I'm going to use `nn.Module` class and create my own model.

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
words = open('names.txt', 'r').read().splitlines()
device

'cuda'

In [3]:
# encode chars to integers
chars = sorted(list(set(''.join(words))))
stoi = { ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0
itos = { i:ch for ch,i in stoi.items()}
vocab_size = len(itos)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [4]:
# hyperparameters
block_size = 16
batch_size = 32

emb_size = 64
n_hidden = 64

In [5]:
# build dataset
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        # sliding window
        for ch in w + '.':
            ix = stoi[ch] # encode(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

n1 = int(0.9*len(words))

X, Y = build_dataset(words)
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:])

def get_batch(split):
    dataX = Xtr if split == 'train' else Xval
    dataY = Ytr if split == 'train' else Yval

    ix = torch.randint(low=0, high=dataX.shape[0], size=(batch_size, ))
    x = torch.stack([dataX[i] for i in ix])
    y = torch.stack([dataY[i] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y


print(X[:5])
print(Y[:5])
len(Xtr)

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13, 13],
        [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 13, 13,  1]])
tensor([ 5, 13, 13,  1,  0])


205411

For fun, I will recreate some boilerplate layers.

In [ ]:
class Flatten(nn.Module):
    def __init__(self):

    def forward(self):
        

In [88]:
class DilatedCasualConvolutions():
    def __init__(self):

    def forward(self):
        

IndentationError: expected an indented block after function definition on line 2 (4139993563.py, line 4)

In [6]:
class Linear(nn.Module):
  def __init__(self, fan_in, fan_out, bias=True):
    super().__init__()
    self.weight = nn.Parameter(torch.randn((fan_in, fan_out)) / fan_in**0.5) # note: kaiming init
    self.bias = nn.Parameter(torch.zeros(fan_out)) if bias else None
  
  def forward(self, x):
    out = x @ self.weight
    if self.bias is not None:
      out += self.bias
    return out

In [117]:
tok_emb = nn.Embedding(vocab_size, emb_size) 
pos_emb = nn.Embedding(block_size, emb_size)
unembed = Linear(block_size * emb_size, vocab_size) # 64, 27
Xb, Yb = get_batch('train') # cuda

tok_emb.to(device)
pos_emb.to(device)
unembed.to(device)

B, T = Xb.shape
tok = tok_emb(Xb) # 32, 16, 64
pos = pos_emb(torch.arange(T, device=device)) # 16, 64
x = tok + pos # 32, 16, 64
x = x.view(x.shape[0], -1) # 32, 16*64
logits = unembed(x) # 32, 27
logits.shape
F.cross_entropy(logits, Yb)

print(Xb[:1])
print(torch.zeros((1, block_size), dtype=int))

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 19,  5]],
       device='cuda:0')
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [250]:
class WaveNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.tok_emb = nn.Embedding(vocab_size, emb_size)
        self.pos_emb = nn.Embedding(block_size, emb_size)
        self.unembed = Linear(block_size * emb_size, vocab_size)

        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.tok_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=device))

        x = tok_emb + pos_emb
        x = x.view(x.shape[0], -1) 
        logits = self.unembed(x)

        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx=None):
        if idx == None:
            idx = torch.zeros((1, block_size), dtype=int, device=device)

        out = []
        print(idx)
        print(idx.shape)
        while True:
            logits, loss = self(idx)
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1).item()

            idx = torch.cat((idx[:, 1:], torch.tensor([[ix]], device=device)), dim=1)

            out.append(decode([ix]))
            if ix == 0:
                break
        return out


In [ ]:
model = WaveNet()
model.to(device=device)

# while True:
#     idx = torch.zeros((1, block_size), dtype=int, device=device)

#     logits, loss = model(idx)
#     probs = F.softmax(logits, dim=1)
#     ix = torch.multinomial(probs, num_samples=1).item()

#     print(decode([ix]))
#     idx = torch.cat((idx[:, 1:], torch.tensor([[ix]], device=device)), dim=1)
#     if ix == 0:
#         break
"".join(model.generate())

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')
torch.Size([1, 16])


'cbljvujvrdworknwvygbgooypacyfgjlciydkvqvlnjntstokgdpkexuiyvpfslydxxmnfppcxuyvuujapfgyrdeiwkeokvlrvuonvvsosruifvqfppwngmzwtswqjsaymjgypcurwwkonwkxdayudi.'

In [254]:
max_iters = 5000
eval_interval = max_iters // 5
lr = 4e-4

model = WaveNet()
model.to(device=device)
print(sum(p.numel() for p in model.parameters()))

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
for iter in range(max_iters):
    Xb, Yb = get_batch('train')
    logits, loss = model(Xb, Yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0 or iter == max_iters - 1:
        print(loss.item())

30427
3.296679973602295
2.3900818824768066
2.320523262023926
2.0314829349517822
2.072024345397949
1.901840329170227


In [259]:
"".join(model.generate())

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0')
torch.Size([1, 16])


'danya.'